# 8. High-Performance Classification with BERT Model (English DistilBERT)

In this notebook, we will go beyond traditional Machine Learning (Logistic Regression, SVM) and standard Deep Learning (TextCNN, FastText) models and use a **Transformer (BERT)** based model, which is the current State-of-the-Art (SOTA) architecture.

**IMPORTANT NOTE:** Since our dataset is massive with 1.6 million reviews, training BERT with all of this data on a standard computer graphics card (GPU) could take days. Therefore:
- We will use the lighter and faster **`distilbert-base-uncased`** model.
- Instead of the entire 1,140,000 training data, we will perform fine-tuning on a **balanced subset of 50,000 or 100,000 reviews**.

If the necessary libraries are not installed, you can run the cell below:

In [ ]:
!pip install transformers datasets accelerate evaluate

In [ ]:
import pandas as pd
import numpy as np
import torch
import evaluate
import joblib
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in use: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data Loading and Subset Creation
We are reducing our massive 1.6 million dataset to RAM and GPU-friendly sizes. When creating the subset, we will ensure that the distribution of classes (Good, Middle, Bad) remains balanced.

In [ ]:
print("Loading data...")
# Save RAM by loading only the necessary columns
df = pd.read_csv('data/reviews_preprocessed.csv', usecols=['text', 'label'])
df = df.dropna(subset=['text'])

# Digitize labels (If not already done)
label_map = {'Bad': 0, 'Middle': 1, 'Good': 2}
df['label_num'] = df['label'].map(label_map)

# --- SUBSET CREATION ---
# Let's select 90,000 data for training and 15,000 for testing (105,000 Total)
SUBSET_SIZE = 105000

if len(df) > SUBSET_SIZE:
    # We take a random sample while preserving the distribution of classes with Stratify
    df_subset, _ = train_test_split(df, train_size=SUBSET_SIZE, stratify=df['label_num'], random_state=42)
else:
    df_subset = df

print(f"Created subset size: {len(df_subset)}")
print(df_subset['label'].value_counts())

In [ ]:
# Splitting into training and test sets
train_df, test_df = train_test_split(df_subset, test_size=0.15, stratify=df_subset['label_num'], random_state=42)

# Converting to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df[['text', 'label_num']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'label_num']].reset_index(drop=True))

## 2. Tokenization (Converting Text to a Format BERT Understands)

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

print("Tokenizing the training set...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing the test set...")
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Setting the column name expected by the HuggingFace model to 'labels'
tokenized_train = tokenized_train.rename_column("label_num", "labels")
tokenized_test = tokenized_test.rename_column("label_num", "labels")

# Converting to PyTorch tensor format
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

## 3. Model Setup and Training (Fine-Tuning)

In [ ]:
# evaluate library metrics for calculating Accuracy and F1 Score
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

# Loading the model (3 Classes: Bad, Middle, Good)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
model.to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir="./bert_logs",
    logging_steps=500,
    fp16=torch.cuda.is_available(), # If GPU is available, accelerates training with 16-bit precision
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

In [ ]:
# Start training
print("Starting BERT Model training...")
trainer.train()

## 4. Evaluation and Saving

In [ ]:
# Final performance on the test set
results = trainer.evaluate()
print("Test Set Results:", results)

# Saving the best model
trainer.save_model("models/distilbert_sentiment_model")
tokenizer.save_pretrained("models/distilbert_sentiment_model")
print("Model successfully saved to 'models/distilbert_sentiment_model' folder!")